# Inverted index + BM25 prototype

Builds an inverted index over a sample of Seattle-area reviews and ranks them with BM25, written from scratch so every step is visible.


In [1]:
import math
import re
from collections import Counter, defaultdict

import pandas as pd
from load_data import load_restaurants, load_reviews

## 1. Sample of reviews
Sample 10000 reviews first

In [2]:
SAMPLE_SIZE = 10_000

restaurants = load_restaurants()
docs = (load_reviews().head(SAMPLE_SIZE)
        .merge(restaurants[["gmap_id", "name", "city"]], on="gmap_id", how="left")
        .reset_index(drop=True))
print(f"{len(docs):,} reviews of {docs['gmap_id'].nunique():,} restaurants")
docs.head()

10,000 reviews of 388 restaurants


,gmap_id,user_id,time,rating,text,name,city
0,0x549015dcda35c8b5:0x59acf0fabf61e550,102172916892203956834,2021-08-14 06:25:37.391,4,This place has a nice ambience. We tried the m...,Citizen Campfire Beer Garden,Seattle
1,0x549015dcda35c8b5:0x59acf0fabf61e550,111339593498069638516,2021-09-06 00:21:29.980,5,As much as I don't want to make this place mor...,Citizen Campfire Beer Garden,Seattle
2,0x549015dcda35c8b5:0x59acf0fabf61e550,105158290017774926865,2021-07-24 22:30:34.474,5,Love this place - adore the chill campfire vibes!,Citizen Campfire Beer Garden,Seattle
3,0x549015dcda35c8b5:0x59acf0fabf61e550,109265099310689872759,2021-08-21 03:06:12.546,2,Stars for the vibes only. Beer was undrinkable...,Citizen Campfire Beer Garden,Seattle
4,0x54906b5052cf8c7d:0xf5182d591ee7748,101250478423909853632,2018-06-18 02:25:06.929,5,Great ambiance but the host is a little odd. 😆,Hansen Central,Seattle


## 2. Tokenizer
- Get rid of extra text from googletranslated reviews that contain `(Translated by Google) <English> (Original) <original language>`
- Make everything lowercase, split on anything that isn't a letter or digit, and drop apostrophes so `don't` → `dont`.
- Dropped stop words, but kept negations (negative reviews)

In [3]:
STOPWORDS = set('''
a an and are as at be been but by for from had has have he her his i if in into is it its
me my of on or our she so than that the their them they this to was we were what when which
who will with you your just also very really there here us did do does
'''.split())

def english_text(text):
    if text.startswith("(Translated by Google)"):
        text = text.removeprefix("(Translated by Google)").split("(Original)")[0]
    return text

def tokenize(text):
    words = re.findall(r"[a-z0-9]+", english_text(text).lower().replace("'", ""))
    return [w for w in words if w not in STOPWORDS]

tokenize("(Translated by Google) Not cozy, but the food's GREAT!\n\n(Original)\nNo es acogedor")

['not', 'cozy', 'foods', 'great']

## 3. Inverted index


In [4]:
index = defaultdict(dict)
doc_len = []

for doc_id, text in enumerate(docs["text"]):
    tokens = tokenize(text)
    doc_len.append(len(tokens))
    for term, tf in Counter(tokens).items():
        index[term][doc_id] = tf

N = len(doc_len)
avgdl = sum(doc_len) / N
print(f"{len(index):,} unique terms, average review length {avgdl:.1f} tokens")
print("postings for 'cozy':", dict(list(index['cozy'].items())[:10]))

12,008 unique terms, average review length 19.2 tokens
postings for 'cozy': {61: 1, 425: 1, 437: 1, 543: 1, 544: 1, 545: 1, 803: 1, 813: 1, 1087: 1, 1093: 1}


In [5]:
# Most common terms by document frequency (number of reviews containing them)
pd.Series({t: len(p) for t, p in index.items()}).sort_values(ascending=False).head(20)

food         3567
good         2388
great        2311
service      1981
place        1747
not          1357
delicious    1271
best         1094
friendly     1036
all           905
staff         868
time          857
like          855
back          806
go            777
nice          751
out           744
amazing       729
one           710
love          705
dtype: int64

## 4. BM25


In [6]:
K1, B = 1.5, 0.75

def idf(term):
    df = len(index.get(term, {}))
    return math.log(1 + (N - df + 0.5) / (df + 0.5))

def bm25(query, k1=K1, b=B):
    scores = defaultdict(float)
    for term in tokenize(query):
        term_idf = idf(term)
        for doc_id, tf in index.get(term, {}).items():
            norm = 1 - b + b * doc_len[doc_id] / avgdl
            scores[doc_id] += term_idf * tf * (k1 + 1) / (tf + k1 * norm)
    return scores

def search(query, k=10, **params):
    scores = bm25(query, **params)
    top = sorted(scores, key=scores.get, reverse=True)[:k]
    results = docs.loc[top, ["name", "city", "rating", "text"]].copy()
    results.insert(0, "score", [round(scores[d], 2) for d in top])
    return results

pd.set_option("display.max_colwidth", 200)
search("cozy hole in the wall")

,score,name,city,rating,text
1971,16.83,Mia's Off Broadway Cafe,Seattle,5,Great hole in the wall with great prices
8409,15.72,Tippe and Drague Alehouse,Seattle,5,Awesome hole in the wall with amazing brunch and great beer.
7325,15.23,Oriental Mart Filipino Restaurant,Seattle,5,Hole in the wall with great food..great conversation. They food is so fresh.
8227,15.23,Taqueria El Rincon Express,Seattle,4,"A hole in the wall kinda place, with authentic affordable priced food."
8436,15.23,Tippe and Drague Alehouse,Seattle,4,Awesome hole in the wall with a great beer selection and awesome sandwiches
8432,14.76,Tippe and Drague Alehouse,Seattle,5,Great hole in the wall bar. Great beer u have never had before.
7016,13.51,I Love Bento's,Kirkland,5,I adore this little hole in the wall restaurant. Great food and a super sweet staff. The remember their customers.
196,12.79,Osaka Grill Teriyaki & Deli,Seattle,4,"This is one of the unchanging landmarks in downtown seattle. Good, cheap teriyaki from a hole-in-the-wall place that's been around forever."
3310,12.46,Huong Que Deli and Cafe,Seattle,4,An authentic cash-only hole-in-the-wall Vietnamese restaurant that serves only a small number of items on the menu. Great food!
3001,11.56,Overcast Coffee Company (Formerly Capital D Cafe),Seattle,5,"Great coffee and the bacon egg and cheese croissant was 👌 cool little hole in the wall place, check it out! My new favorite neighborhood coffee shop"


In [7]:
search("quiet romantic date night")

,score,name,city,rating,text
4194,12.91,Olive You,Kirkland,4,"Great atmosphere - they have two sections, one with a bar and more of a night life feel, the other a more romantic dining area."
119,12.62,Old Salt @ Manolin,Seattle,5,"Fantastic food, specialty drinks, super friendly staff. Loved this place for date night!!"
5379,11.02,Bizzarro Italian Cafe - White Center,Seattle,5,"We had a great date night here, The food was excellent it's an Italian inspired experience with some Pacific Northwest flair and the staff is super friendly and attentive!"
9852,9.65,Zio Sal Ristorante,Redmond,5,Amazing food! I love coming here. So romantic and very good prices! ❤️ you should come!
473,9.51,Chez Shea,Seattle,5,a great date spot.
292,9.36,Field to Table,Seattle,5,"One of the best experiences I’ve ever done on a whim, date night was blown out the water with the amazing food and staff. Dessert was being able to kick a field goal at Lumen field."
796,9.16,Aoki Sushi & Grill,Seattle,5,Had a wonderful birthday there food was hot and came in a timely manner! Also great place for a date night or small outing. Place was clean and atmosphere was warm and welcoming ☺. I highly enjoye...
7815,9.16,Roaring Bowl,Seattle,5,Great date night spot! Husband took me and it was awesome. The staff was super cool and fun. They even let us stay and finish our meal instead of rushing us out after close. I will def be going ba...
3040,8.98,Baked in Bosnia,Seattle,5,"Absolutely fantastic!! Made for a really fun surprise date night. Everything was really delicious and I can't wait to order again! The pita was to die for, the gulash was so dang good, and the bak..."
1533,8.77,Fine Food Breakfast & Lunch,Federal Way,4,Fairly quiet. Lots of seating.


## 5. Searching through the ambience word lists
Repeated words are only counted once.

In [8]:
import json

with open("data/ambience.json") as f:
    ambience_terms = json.load(f)

search(" ".join(ambience_terms["cozy"]), k=5)

,score,name,city,rating,text
8864,16.74,Starbucks,Kent,5,"Awesome as usual ❤, staff are very welcoming and the chairs so comfy with a lot of choices!!"
357,15.57,Polaris Restaurant,Bellevue,5,"They have a great selection on the menu, the food came out quickly and it was very healthy and flavorful. Their breakfast is amazing as well and the bar is very quaint and comfy."
9228,14.88,Homegrown,Seattle,5,They donate sandwhiches to all our volunteers for all.of our events and their sandwhiches are so welcoming and comforting especially working at busy events.
8488,14.82,Tippe and Drague Alehouse,Seattle,5,"Great beer menu, super friendly service. The interior is a bit small but intimate and cozy."
2325,13.49,Caffè Musica,Seattle,5,"Cozy, homey, delicious in house made pasta really friendly staff. Some nights they do music still very thoughtful of guests, and relaxing."


## 6. From reviews to restaurants
Sum each restaurant's review scores, so a place many reviewers call cozy ranks above one with a single cozy review.

In [9]:
def search_restaurants(query, k=10):
    scores = pd.Series(bm25(query), dtype=float)
    by_place = (docs.loc[scores.index, ["gmap_id", "name", "city"]]
                .assign(score=scores.values)
                .groupby(["gmap_id", "name", "city"])["score"]
                .agg(total="sum", matching_reviews="count")
                .sort_values("total", ascending=False))
    return by_place.head(k).reset_index().drop(columns="gmap_id")

search_restaurants(" ".join(ambience_terms["cozy"]))

,name,city,total,matching_reviews
0,Tippe and Drague Alehouse,Seattle,96.643390,13
1,Connect Lounge,Seattle,58.660164,11
2,Good Neighbor Cafe,Seattle,57.206237,8
3,Subway,Seattle,52.128058,7
4,Starbucks,Kent,41.207762,5
5,Caffè Musica,Seattle,34.369909,5
6,Aoki Sushi & Grill,Seattle,31.590633,6
7,Zio Sal Ristorante,Redmond,31.397179,4
8,Dunbar Room at Hotel Sorrento,Seattle,29.163611,5
9,Al Boccalino,Seattle,28.018240,5
